# [Assignment 3: Data Integration & Data Preparation](https://hqahtan.github.io/Teaching/infomdw/assignments/Assignment_3_26_27.html)



In [2]:
%pip install -q polars Bio

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import polars as pl
from Bio.Align import PairwiseAligner
from rapidfuzz.distance import Jaro, Levenshtein
from rapidfuzz.process import cpdist

## Task 1: Record Matching 

In [4]:
# Load ACM and DBLP2 datasets into pandas DataFrames
acm = pl.read_csv("./raw/acm.csv", encoding="ISO-8859-1")
dblp2 = pl.read_csv("./raw/DBLP2.csv", encoding="ISO-8859-1")
eval_mappings = pl.read_csv("./raw/DBLP-ACM_perfectMapping.csv", encoding="ISO-8859-1")

# rename columns
eval_mappings = eval_mappings.rename({"idDBLP": "id_dblp", "idACM": "id_acm"})

### Display the data

In [5]:
acm.head()

id,title,authors,venue,year
i64,str,str,str,i64
304586,"""The WASA2 object-oriented work…","""Gottfried Vossen, Mathias Wesk…","""International Conference on Ma…",1999
304587,"""A user-centered interface for …","""Isabel F. Cruz, Kimberly M. Ja…","""International Conference on Ma…",1999
304589,"""World Wide Database-integratin…","""Athman Bouguettaya, Boualem Be…","""International Conference on Ma…",1999
304590,"""XML-based information mediatio…","""Chaitan Baru, Amarnath Gupta, …","""International Conference on Ma…",1999
304582,"""The CCUBE constraint object-or…","""Alexander Brodsky, Victor E. S…","""International Conference on Ma…",1999


In [6]:
dblp2.head()

id,title,authors,venue,year
str,str,str,str,i64
"""journals/sigmod/Mackay99""","""Semantic Integration of Enviro…","""D. Scott Mackay""","""SIGMOD Record""",1999
"""conf/vldb/PoosalaI96""","""Estimation of Query-Result Dis…","""Viswanath Poosala, Yannis E. I…","""VLDB""",1996
"""conf/vldb/PalpanasSCP02""","""Incremental Maintenance for No…","""Themistoklis Palpanas, Richard…","""VLDB""",2002
"""conf/vldb/GardarinGT96""","""Cost-based Selection of Path E…","""Zhao-Hui Tang, Georges Gardari…","""VLDB""",1996
"""conf/vldb/HoelS95""","""Benchmarking Spatial Join Oper…","""Erik G. Hoel, Hanan Samet""","""VLDB""",1995


### Data cleaning

In [7]:
def clean_strings(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        # Only apply the following string cleaning operations on columns of type String
        pl.col(pl.String)
        # 1. Change all alphabetical characters into lowercase for Pandas dataframe values
        .str.to_lowercase()
        # 2. Eliminate extra spaces by trimming leading and trailing spaces
        .str.strip_chars()
        # 3. Replace consecutive whitespaces with a single space
        .str.replace_all(r"\s+", " ")
    )


# Perform string cleaning on the datasets
dblp2 = clean_strings(dblp2)
acm = clean_strings(acm)
eval_mappings = clean_strings(eval_mappings)

### Levenshtein similarity

In [8]:
# %%timeit

# Select and alias title columns from ACM and DBLP datasets
acm_titles = acm.select(pl.col("title").alias("title_acm"))
dblp_titles = dblp2.select(pl.col("title").alias("title_dblp"))

# Create a cartesian product between ACM and DBLP titles
acm_x_dblp_titles = acm_titles.join(dblp_titles, how="cross")

# Calculate distances and similarities between ACM and DBLP titles
acm_x_dblp_titles = acm_x_dblp_titles.with_columns(
    # Calculate Levenshtein distance
    ldist=pl.Series(
        cpdist(
            acm_x_dblp_titles["title_acm"],
            acm_x_dblp_titles["title_dblp"],
            scorer=Levenshtein.distance,
            dtype="uint32",
            workers=-1,
        )
    ),
    # Calculate normalized similarity
    lsim=pl.Series(
        cpdist(
            acm_x_dblp_titles["title_acm"],
            acm_x_dblp_titles["title_dblp"],
            scorer=Levenshtein.normalized_similarity,
            dtype="float64",
            workers=-1,
        )
    ),
)

acm_x_dblp_titles.head()

title_acm,title_dblp,ldist,lsim
str,str,u32,f64
"""the wasa2 object-oriented work…","""semantic integration of enviro…",83,0.245455
"""the wasa2 object-oriented work…","""estimation of query-result dis…",73,0.197802
"""the wasa2 object-oriented work…","""incremental maintenance for no…",50,0.21875
"""the wasa2 object-oriented work…","""cost-based selection of path e…",69,0.233333
"""the wasa2 object-oriented work…","""benchmarking spatial join oper…",50,0.107143


### Jaro similarity

In [9]:
# %%timeit

# Select and alias authors columns from ACM and DBLP datasets
acm_authors = acm.select(pl.col("authors").alias("authors_acm"))
dblp_authors = dblp2.select(pl.col("authors").alias("authors_dblp"))

# Create a cartesian product between ACM and DBLP authors
acm_x_dblp_authors = acm_authors.join(dblp_authors, how="cross")

# Compute Jaro similarity between ACM and DBLP authors
acm_x_dblp_authors = acm_x_dblp_authors.with_columns(
    jsim=pl.Series(
        cpdist(
            acm_x_dblp_authors["authors_acm"].to_list(),
            acm_x_dblp_authors["authors_dblp"].to_list(),
            scorer=Jaro.similarity,
            dtype="float64",
            workers=-1,
        )
    )
)

acm_x_dblp_authors.head()

authors_acm,authors_dblp,jsim
str,str,f64
"""gottfried vossen, mathias wesk…","""d. scott mackay""",0.563082
"""gottfried vossen, mathias wesk…","""viswanath poosala, yannis e. i…",0.56395
"""gottfried vossen, mathias wesk…","""themistoklis palpanas, richard…",0.574531
"""gottfried vossen, mathias wesk…","""zhao-hui tang, georges gardari…",0.571552
"""gottfried vossen, mathias wesk…","""erik g. hoel, hanan samet""",0.637252


### Affine-gap similarity

In [10]:
# %%timeit

# Select and alias venue columns from ACM and DBLP datasets
acm_venue = acm.select(pl.col("venue").alias("venue_acm"))
dblp_venue = dblp2.select(pl.col("venue").alias("venue_dblp"))

# Create a cartesian product between ACM and DBLP venues
acm_x_dblp_venue = acm_venue.join(dblp_venue, how="cross")

# Initialize the pairwise aligner for affine gap scoring
aligner = PairwiseAligner(
    mode="global",
    match_score=0,
    mismatch_score=-2,
    open_gap_score=-1,
    extend_gap_score=-0.1,
)

acm_x_dblp_venue = acm_x_dblp_venue.with_columns(
    affine_dist=pl.Series(
        -aligner.score(a, b)
        for a, b in zip(acm_x_dblp_venue["venue_acm"], acm_x_dblp_venue["venue_dblp"])
    )
)

acm_x_dblp_venue = acm_x_dblp_venue.with_columns(
    asim=1
    - 1
    / (
        1
        + pl.col("affine_dist")
        / pl.max_horizontal(
            pl.col("venue_acm").str.len_chars(),
            pl.col("venue_dblp").str.len_chars(),
        ).clip(lower_bound=1)
    )
)

acm_x_dblp_venue.head()

venue_acm,venue_dblp,affine_dist,asim
str,str,f64,f64
"""international conference on ma…","""sigmod record""",7.7,0.143389
"""international conference on ma…","""vldb""",6.8,0.128788
"""international conference on ma…","""vldb""",6.8,0.128788
"""international conference on ma…","""vldb""",6.8,0.128788
"""international conference on ma…","""vldb""",6.8,0.128788


### Match year

In [11]:
# %%timeit

# Select and alias title columns from ACM and DBLP datasets
acm_year = acm.select(pl.col("year").alias("year_acm"))
dblp_year = dblp2.select(pl.col("year").alias("year_dblp"))

# Create a cartesian product between ACM and DBLP titles
acm_x_dblp_year = acm_year.join(dblp_year, how="cross")

# Compute a binary similarity column for the year
acm_x_dblp_year = acm_x_dblp_year.with_columns(
    bsim=(pl.col("year_acm") == pl.col("year_dblp")).cast(pl.Int8)
)

acm_x_dblp_year.head()

year_acm,year_dblp,bsim
i64,i64,i8
1999,1999,1
1999,1996,0
1999,2002,0
1999,1996,0
1999,1995,0


### Final combined similarity score

In [12]:
# Weights: all > 0 and sum to 1
W_TITLE, W_AUTHORS, W_VENUE, W_YEAR = 0.35, 0.25, 0.15, 0.25
assert abs(W_TITLE + W_AUTHORS + W_VENUE + W_YEAR - 1) < 1e-9

# All frames are cross joins of acm x dblp2 in the same row order,
# so their columns line up row by row.
ids = acm.select(pl.col("id").alias("id_acm")).join(
    dblp2.select(pl.col("id").alias("id_dblp")), how="cross"
)

# Calculate weighted similarity for each pair of ACM and DBLP entries
acm_x_dblp_wsim = pl.concat(
    [
        ids,
        acm_x_dblp_titles.select("lsim"),
        acm_x_dblp_authors.select("jsim"),
        acm_x_dblp_venue.select("asim"),
        acm_x_dblp_year.select("bsim"),
    ],
    how="horizontal",
).with_columns(
    simrec=W_TITLE * pl.col("lsim")
    + W_AUTHORS * pl.col("jsim")
    + W_VENUE * pl.col("asim")
    + W_YEAR * pl.col("bsim")
)


In [13]:
acm_x_dblp_wsim

id_acm,id_dblp,lsim,jsim,asim,bsim,simrec
i64,str,f64,f64,f64,i8,f64
304586,"""journals/sigmod/mackay99""",0.245455,0.563082,0.143389,1,0.498188
304586,"""conf/vldb/poosalai96""",0.197802,0.56395,0.128788,0,0.229537
304586,"""conf/vldb/palpanasscp02""",0.21875,0.574531,0.128788,0,0.239514
304586,"""conf/vldb/gardaringt96""",0.233333,0.571552,0.128788,0,0.243873
304586,"""conf/vldb/hoels95""",0.107143,0.637252,0.128788,0,0.216131
…,…,…,…,…,…,…
672979,"""journals/tods/karpsp03""",0.220588,0.523601,0.203822,0,0.238679
672979,"""conf/vldb/limwv03""",0.242857,0.616602,0.163347,0,0.263653
672979,"""journals/tods/chakrabartikmp02""",0.231707,0.538949,0.203822,0,0.246408


In [14]:
ids.head()

id_acm,id_dblp
i64,str
304586,"""journals/sigmod/mackay99"""
304586,"""conf/vldb/poosalai96"""
304586,"""conf/vldb/palpanasscp02"""
304586,"""conf/vldb/gardaringt96"""
304586,"""conf/vldb/hoels95"""


In [15]:
# Keep mappings with similarity greater than 0.7 and sort by id_acm
candidate_mappings = acm_x_dblp_wsim.filter(pl.col("simrec") > 0.7).sort("id_acm")

# If there are multiple mappings for the same ACM entry, keep only the one with the highest similarity
# Keep only the highest-similarity match for each ACM record
mappings = (
    candidate_mappings.sort("simrec", descending=True)
    .unique(subset="id_acm", keep="first", maintain_order=True)
    .sort("id_acm")
)

### Check mappings with validation set

In [16]:
# Find true positives by joining the predicted and evaluation mappings
comparison = mappings.join(eval_mappings, on=["id_dblp", "id_acm"], how="inner")

# Create confusion matrix
tp = comparison.height  # reported and truly duplicates
fp = eval_mappings.height - tp  # reported but not duplicates
fn = mappings.height - tp  # true duplicates that were missed
tn = acm.height * dblp2.height - tp - fp - fn  # all other pairs

confusion = pl.DataFrame(
    {
        "": ["Actual duplicate", "Actual non-duplicate"],
        "Predicted duplicate": [tp, fp],
        "Predicted non-duplicate": [fn, tn],
    }
)
print(confusion)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

shape: (2, 3)
┌──────────────────────┬─────────────────────┬─────────────────────────┐
│                      ┆ Predicted duplicate ┆ Predicted non-duplicate │
│ ---                  ┆ ---                 ┆ ---                     │
│ str                  ┆ i64                 ┆ i64                     │
╞══════════════════════╪═════════════════════╪═════════════════════════╡
│ Actual duplicate     ┆ 2158                ┆ 23                      │
│ Actual non-duplicate ┆ 66                  ┆ 5998857                 │
└──────────────────────┴─────────────────────┴─────────────────────────┘
Precision: 0.9703, Recall: 0.9895


### Profiling

We use [timeit](https://docs.python.org/3/library/timeit.html) to record the execution time of the similarity algorithms and the required data transformations.

System specifications:
- Processor: Intel(R) Core(TM) i7-10850H CPU @ 2.70GHz (2.71 GHz)
- RAM: 16.0 GB (15.6 GB usable)
- Graphics card: NVIDIA GeForce MX250 (2 GB)

Execution times:

| Levenshtein | Jaro | Affine gap | Match |
|---|---|---|---|
| 5.17 s ± 99.2 ms | 2.37 s ± 61.5 ms | 40.8 s ± 677 ms | 41 ms ± 1.29 ms |

Values are mean ± standard deviation of 7 runs, 1 loop each.

Total mean execution time: **48.381 seconds**

### Discussion

The methods we used compares every ACM record with every DBLP record, producing a Cartesian product exceeding 6 million pairs to compare. We quickly realized that this could pose as a bottleneck during our development. To maintain fast execution times for debugging and evaluation, we optimized our code by integrating fast Rust based libraries such as Polars and RapidFuzz. We did have one slow method which was the affine-gap calculation taking 40.8 seconds, most of the total execution time (48.4 seconds).

There are several ways to reduce the execution time:

1. We could make better use of the hardware. CPython is considered one of the most poorly optimized programming languages out there, as it doesn't use the hardware to its full power. This is because of its interpreted nature and the fact that most of the code is still executed single-threaded (slowly being fixed in the latest versions). By using a compiled language like Rust, or multi-threading, we can make the execution time faster without changing the logic. The only disadvantage is that it requires more work.

2. We could reduce the product size. By grouping records into candidate pairs using one of the other similarity methods, we can avoid evaluating pairs with no chance of matching. However, we have to be careful here, if we don't create the right rules, we can miss true matches.

3. We could also perform MinHashing combined with Locality-Sensitive Hashing (LSH) to quickly identify candidate pairs before running computationally expensive algorithms like Affine-gap distance. This has the same effect as the previous solution, in the sense that it results in fewer pairs to compare. However, just like that solution, it could result in us missing true matches.

## Task 3: Data profiling and preparation

Dataset: one collision file from the UK road safety open data (`dft-road-casualty-statistics-collision-<year>.csv`), placed in `raw/`.

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

DATA_DIR = Path("raw")
csv_path = sorted(DATA_DIR.glob("dft-road-casualty-statistics-collision-*.csv"))[-1]
print("Using", csv_path)

# df: values parsed by pandas (numbers as numbers, empty/"NULL" as NaN)
# df_str: every value as the original text, used for type, length and pattern statistics
df = pd.read_csv(csv_path, low_memory=False)
df_str = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
print(df.shape)
df.head()

Using raw/dft-road-casualty-statistics-collision-2025.csv
(101525, 44)


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,202517M101925,2025,17M101925,447673.0,519279.0,-1.26420,54.56650,17,3,1,...,-1,0,1,1,2,E01012014,3,1,0.0,1.0
1,202517L205225,2025,17L205225,454611.0,518597.0,-1.15704,54.55967,17,3,2,...,-1,0,1,1,2,E01012173,3,1,0.0,1.0
2,2025070971700,2025,070971700,354476.0,382656.0,-2.68516,53.33884,7,3,2,...,-1,0,2,1,2,E01012444,3,1,0.0,1.0
3,2025070403274,2025,070403274,361793.0,393163.0,-2.57655,53.43386,7,3,2,...,-1,0,2,3,2,E01012468,3,1,0.0,1.0
4,2025071000604,2025,071000604,366083.0,389327.0,-2.51157,53.39967,7,3,7,...,-1,0,2,1,1,E01012551,3,1,0.0,1.0


### 3.1 Data profiling: summary statistics

Statistics chosen from the single-column and multi-column profiling tasks in Abedjan, Golab & Naumann, *Profiling relational data: a survey* (VLDB Journal, 2015).

#### Statistics 1-5: number of rows, null values, distinct values, uniqueness, constancy (cardinalities and value distribution)

In [2]:
n_rows = len(df)
print("Statistic 1 - number of rows:", n_rows, "| number of columns:", df.shape[1])

profile = pd.DataFrame(index=df.columns)
# 2. null values: cells that are explicitly empty / NULL in the file
profile["n_null"] = df.isna().sum()
profile["pct_null"] = (100 * profile["n_null"] / n_rows).round(2)
# 3. distinct values (cardinality of the column)
profile["n_distinct"] = df.nunique(dropna=True)
# 4. uniqueness = distinct / rows; 1.0 means the column is a candidate key
profile["uniqueness"] = (profile["n_distinct"] / n_rows).round(4)
# 5. constancy = frequency of the most frequent value / rows
profile["most_frequent"] = df.apply(lambda c: c.mode(dropna=True).iloc[0] if c.notna().any() else None)
profile["constancy"] = (df.apply(lambda c: c.value_counts(dropna=True).iloc[0] if c.notna().any() else 0) / n_rows).round(4)
profile.sort_values("pct_null", ascending=False)

Statistic 1 - number of rows: 101525 | number of columns: 44


,n_null,pct_null,n_distinct,uniqueness,most_frequent,constancy
collision_index,0,0.0,101525,1.0000,2025010551784,0.0000
collision_year,0,0.0,1,0.0000,2025,1.0000
junction_control,0,0.0,6,0.0001,-1,0.4382
second_road_class,0,0.0,9,0.0001,6,0.5549
second_road_number,0,0.0,2487,0.0245,0,0.5825
pedestrian_crossing_human_control_historic,0,0.0,5,0.0000,-1,0.8653
pedestrian_crossing_physical_facilities_historic,0,0.0,8,0.0001,-1,0.8653
pedestrian_crossing,0,0.0,10,0.0001,0,0.7342
light_conditions,0,0.0,6,0.0001,1,0.7172
weather_conditions,0,0.0,9,0.0001,1,0.8215


#### Statistics 6-7: basic type and value length (data types and domains)

In [ ]:
def basic_type(values):
    """Classify a text column as numeric, date, time, alphabetic or alphanumeric (most common class wins)."""
    v = values[values.str.len() > 0]
    if v.empty:
        return "empty"
    classes = np.select(
        [
            v.str.fullmatch(r"-?\d+(\.\d+)?"),
            v.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}|\d{4}-\d{2}-\d{2}"),
            v.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?"),
            v.str.fullmatch(r"[A-Za-z ]+"),
        ],
        ["numeric", "date", "time", "alphabetic"],
        default="alphanumeric",
    )
    return pd.Series(classes).value_counts().idxmax()


lengths = df_str.apply(lambda c: c.str.len())
profile["basic_type"] = df_str.apply(basic_type)
profile["pandas_dtype"] = df.dtypes.astype(str)
profile["min_len"] = lengths.min()
profile["max_len"] = lengths.max()
profile["avg_len"] = lengths.mean().round(2)
profile[["basic_type", "pandas_dtype", "min_len", "max_len", "avg_len"]]

#### Statistic 8: minimum, maximum and quartiles (value distribution) of the numeric columns

In [ ]:
numeric_cols = df.select_dtypes("number").columns
quartiles = df[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75]).T
quartiles[["min", "25%", "50%", "75%", "max", "mean", "std"]].round(3)

#### Statistic 9: histograms (value distribution)

In [ ]:
hist_cols = ["number_of_vehicles", "number_of_casualties", "speed_limit", "latitude"]
hist_cols = [c for c in hist_cols if c in df.columns]

fig, axes = plt.subplots(1, len(hist_cols), figsize=(4 * len(hist_cols), 3))
for ax, col in zip(np.atleast_1d(axes), hist_cols):
    df[col].dropna().plot.hist(bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# Frequency table for the low-cardinality columns (shows the coded values, including -1 / 99)
for col in ["number_of_vehicles", "speed_limit"]:
    if col in df.columns:
        print(df[col].value_counts(dropna=False).sort_index().to_string(), "\n")

#### Statistic 10: number of decimals (data types and domains)

In [ ]:
def max_decimals(values):
    """Largest number of digits after the decimal point among numeric text values."""
    num = values[values.str.fullmatch(r"-?\d+(\.\d+)?")]
    return int(num.str.split(".").str[1].fillna("").str.len().max()) if len(num) else 0


profile["max_decimals"] = df_str.apply(max_decimals)
profile.loc[profile["basic_type"] == "numeric", ["max_decimals"]].T

#### Statistic 11: value patterns (data types and domains)

In [ ]:
def shape(s):
    """Replace digits by 9 and letters by A, e.g. 'E01000123' -> 'A99999999'."""
    return re.sub(r"[A-Za-z]", "A", re.sub(r"\d", "9", s))


pattern_cols = [c for c in ["collision_index", "date", "time", "lsoa_of_accident_location",
                            "local_authority_ons_district", "location_easting_osgr"] if c in df_str.columns]
for col in pattern_cols:
    pats = df_str[col].map(shape).value_counts()
    print(f"{col}: {len(pats)} distinct pattern(s)")
    print(pats.head(5).to_string(), "\n")

#### Statistic 12: first-digit distribution (value distribution, Benford's law)

In [ ]:
benford = pd.Series(np.log10(1 + 1 / np.arange(1, 10)), index=range(1, 10))

col = "first_road_number"
vals = df[col][df[col] > 0].astype(int).astype(str)  # 0 and -1 are codes, not road numbers
first_digit = vals.str[0].astype(int).value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({"observed": first_digit, "benford": benford}).round(3)
comparison.plot.bar(figsize=(7, 3), title=f"First digit of {col}")
plt.show()
comparison.T

#### Statistic 13: unique column combinations (candidate keys, multi-column dependencies)

In [ ]:
# Single columns that are unique (and have no nulls) are candidate keys
single_keys = profile.index[(profile["uniqueness"] == 1.0) & (profile["n_null"] == 0)].tolist()
print("Unique single columns:", single_keys)

# Check whether a few natural column combinations are also unique
for combo in [["collision_year", "collision_ref_no"], ["police_force", "date", "time"]]:
    if all(c in df.columns for c in combo):
        dup = df.duplicated(subset=combo).sum()
        print(combo, "-> unique" if dup == 0 else f"-> not unique ({dup} duplicate rows)")

#### Statistic 14: functional dependencies (multi-column dependencies)

In [ ]:
def fd_violations(data, lhs, rhs):
    """Share of rows whose lhs value maps to more than one rhs value (0 means lhs -> rhs holds)."""
    n_rhs = data.groupby(lhs)[rhs].nunique()
    bad = n_rhs[n_rhs > 1].index
    return data[lhs].isin(bad).mean()


for lhs, rhs in [("date", "day_of_week"),
                 ("local_authority_ons_district", "police_force"),
                 ("lsoa_of_accident_location", "local_authority_ons_district"),
                 ("speed_limit", "urban_or_rural_area")]:
    if lhs in df.columns and rhs in df.columns:
        v = fd_violations(df, lhs, rhs)
        print(f"{lhs} -> {rhs}: {'holds' if v == 0 else f'violated by {100 * v:.2f}% of rows'}")